We follow the guideline https://herbie.readthedocs.io/en/latest/gallery/noaa_models/hrrr.html to read HRRR data

In [1]:
import sys
print(sys.executable)

C:\Users\chenk\Desktop\GSRCONUS\.venv\Scripts\python.exe


In [5]:
from herbie import Herbie
from herbie.toolbox import EasyMap, pc
from herbie import paint, FastHerbie
import matplotlib.pyplot as plt
import xarray as xr
import h3
import gc
from pathlib import Path
import numpy as np
import pandas as pd
xr.set_options(use_new_combine_kwarg_defaults=True)

# Get H3 index

Since the lat/lon grid is static over time (same points for all timesteps), we only need to map to H3 once to save computation. Here I chose resolution = 5.

## Evidence

In [7]:
FH = FastHerbie(
    pd.date_range("2021-01-01", "2021-01-03 23:00", freq="1h"),
    model="hrrr",
    product="sfc",
    fxx=[0],
)

ds = FH.xarray("TMP:2 m above ground")

print(ds.time.values)

['2021-01-01T00:00:00.000000000' '2021-01-01T01:00:00.000000000'
 '2021-01-01T02:00:00.000000000' '2021-01-01T03:00:00.000000000'
 '2021-01-01T04:00:00.000000000' '2021-01-01T05:00:00.000000000'
 '2021-01-01T06:00:00.000000000' '2021-01-01T07:00:00.000000000'
 '2021-01-01T08:00:00.000000000' '2021-01-01T09:00:00.000000000'
 '2021-01-01T10:00:00.000000000' '2021-01-01T11:00:00.000000000'
 '2021-01-01T12:00:00.000000000' '2021-01-01T13:00:00.000000000'
 '2021-01-01T14:00:00.000000000' '2021-01-01T15:00:00.000000000'
 '2021-01-01T16:00:00.000000000' '2021-01-01T17:00:00.000000000'
 '2021-01-01T18:00:00.000000000' '2021-01-01T19:00:00.000000000'
 '2021-01-01T20:00:00.000000000' '2021-01-01T21:00:00.000000000'
 '2021-01-01T22:00:00.000000000' '2021-01-01T23:00:00.000000000'
 '2021-01-02T00:00:00.000000000' '2021-01-02T01:00:00.000000000'
 '2021-01-02T02:00:00.000000000' '2021-01-02T03:00:00.000000000'
 '2021-01-02T04:00:00.000000000' '2021-01-02T05:00:00.000000000'
 '2021-01-02T06:00:00.000

In [9]:
times = ds.time.values

for t, ts in enumerate(times):
    print(t, ts)

0 2021-01-01T00:00:00.000000000
1 2021-01-01T01:00:00.000000000
2 2021-01-01T02:00:00.000000000
3 2021-01-01T03:00:00.000000000
4 2021-01-01T04:00:00.000000000
5 2021-01-01T05:00:00.000000000
6 2021-01-01T06:00:00.000000000
7 2021-01-01T07:00:00.000000000
8 2021-01-01T08:00:00.000000000
9 2021-01-01T09:00:00.000000000
10 2021-01-01T10:00:00.000000000
11 2021-01-01T11:00:00.000000000
12 2021-01-01T12:00:00.000000000
13 2021-01-01T13:00:00.000000000
14 2021-01-01T14:00:00.000000000
15 2021-01-01T15:00:00.000000000
16 2021-01-01T16:00:00.000000000
17 2021-01-01T17:00:00.000000000
18 2021-01-01T18:00:00.000000000
19 2021-01-01T19:00:00.000000000
20 2021-01-01T20:00:00.000000000
21 2021-01-01T21:00:00.000000000
22 2021-01-01T22:00:00.000000000
23 2021-01-01T23:00:00.000000000
24 2021-01-02T00:00:00.000000000
25 2021-01-02T01:00:00.000000000
26 2021-01-02T02:00:00.000000000
27 2021-01-02T03:00:00.000000000
28 2021-01-02T04:00:00.000000000
29 2021-01-02T05:00:00.000000000
30 2021-01-02T06:00:

In [3]:
H1 = Herbie(
    "2021-07-19",
    model="hrrr",
    product="sfc",
    fxx=0,
)
H2 = Herbie(
    "2022-07-30",
    model="hrrr",
    product="sfc",
    fxx=0,
) # randomly select any two days

ds_u1 = H1.xarray("UGRD:10 m above ground")
ds_u2 = H2.xarray("UGRD:10 m above ground")
lat1 = ds_u1.latitude.values
lat2 = ds_u2.latitude.values
np.allclose(lat1, lat2) # if our conclusion is correct, it should output True

✅ Found ┊ model=hrrr ┊ product=sfc ┊ 2021-Jul-19 00:00 UTC F00 ┊ GRIB2 @ aws ┊ IDX @ local
✅ Found ┊ model=hrrr ┊ product=sfc ┊ 2022-Jul-30 00:00 UTC F00 ┊ GRIB2 @ aws ┊ IDX @ local


True

## H3 Index

In [4]:
H = Herbie(
    "2020-01-01",
    model="hrrr",
    product="sfc",
    fxx=0,
)

✅ Found ┊ model=hrrr ┊ product=sfc ┊ 2020-Jan-01 00:00 UTC F00 ┊ GRIB2 @ aws ┊ IDX @ local


In [5]:
ds_u = H.xarray("UGRD:10 m above ground")
lats = ds_u.latitude.values
lons = ds_u.longitude.values

res = 5 # use resolution 5,

h3_index = np.vectorize(lambda lat, lon: h3.latlng_to_cell(lat, lon, res))(lats, lons)

h3_flat = h3_index.ravel()
h3_unique, h3_codes = np.unique(h3_flat, return_inverse=True)

np.save("hrrr_h3_unique.npy", h3_unique)
np.save("hrrr_h3_codes.npy", h3_codes.astype(np.int32)) # save index

# FastHerbie to save data

In [15]:
h3_codes = np.load("hrrr_h3_codes.npy")
h3_unique = np.load("hrrr_h3_unique.npy")
n_h3 = h3_unique.shape[0]
h3_counts = np.bincount(h3_codes, minlength=n_h3)

In [16]:
def h3_max(values_2d, h3_codes, n_h3):
    v = values_2d.ravel()
    mask = np.isfinite(v)
    m = np.full(n_h3, -np.inf)
    np.maximum.at(m, h3_codes[mask], v[mask])
    m[m == -np.inf] = np.nan
    return m


In [10]:
def process_batch(ds_sfc, ds_prs, ds_TMP, ds_MSLMA, dates_batch):

    n_time = min(
        ds_prs.sizes["time"],
        ds_sfc.sizes["time"],
        ds_TMP.sizes["time"],
        ds_MSLMA.sizes["time"]
    )

    for t in range(n_time):

        ts = pd.Timestamp(ds_prs.time.values[t])

        hour_data = {}

        for var in ds_sfc.data_vars:
            arr = ds_sfc[var].isel(time=t).values
            hour_data[var] = h3_max(arr, h3_codes, n_h3)

        for var in ds_TMP.data_vars:
            arr = ds_TMP[var].isel(time=t).values
            hour_data[var] = h3_max(arr, h3_codes, n_h3)

        for var in ds_MSLMA.data_vars:
            arr = ds_MSLMA[var].isel(time=t).values
            hour_data[var] = h3_max(arr, h3_codes, n_h3)

        for var in ds_prs.data_vars:
            da = ds_prs[var]

            if "isobaricInhPa" not in da.dims:
                continue

            arr = da.isel(time=t).values
            levels = da.isobaricInhPa.values

            for lev_i, lev in enumerate(levels):

                name = f"{var}_{lev}"

                hour_data[name] = h3_max(
                    arr[lev_i],
                    h3_codes,
                    n_h3
                )

        hour_data["h3"] = h3_unique

        write_parquet(hour_data, ts)

In [12]:
import pyarrow as pa
import pyarrow.parquet as pq
import os

def write_parquet(hour_data, ts):

    year_folder = ts.strftime("%Y")
    month_folder = ts.strftime("%m")
    day_folder = ts.strftime("%d")
    hour_name = ts.strftime("%H")

    out_dir = f"output/{year_folder}/{month_folder}/{day_folder}"
    os.makedirs(out_dir, exist_ok=True)

    path = f"{out_dir}/{hour_name}.parquet"

    table = pa.table(hour_data)
    pq.write_table(table, path)

    print("write:", path)

In [13]:
def ensure_dataset(ds):
    if isinstance(ds, list):
        flat = []
        for x in ds:
            if isinstance(x, list):
                flat.extend(x)
            else:
                flat.append(x)
        ds = xr.merge(flat, join="outer")
    return ds

In [ ]:
import time

all_dates = pd.date_range(
    start="2021-01-01 00:00",
    end="2021-01-05 23:00",
    freq="1h"
) # use ten days as example

print(f"Total hours to process: {len(all_dates)}")

batch_size = 24

all_results = []
all_names = []
t_global_start = time.time()

for i in range(0, len(all_dates), batch_size):

    dates_batch = all_dates[i : i + batch_size]

    batch_start = dates_batch[0]
    batch_end = dates_batch[-1]

    print()
    print(f"Processing batch {(i // batch_size) + 1}")
    print(f"Time range: From {batch_start} to {batch_end}")

    t_batch_start = time.time()
    FH = FastHerbie(
        dates_batch,
        model="hrrr",
        product="sfc",
        fxx=[0],
        max_threads=8,
    )
    surface_regex = (
        "UGRD:10 m above ground|"
        "VGRD:10 m above ground"
    )

    ds_sfc = FH.xarray(surface_regex)
    ds_TMP = FH.xarray("TMP:2 m above ground")
    ds_MSLMA = FH.xarray("MSLMA:mean sea level")

    prs_vars = [
        "HGT:50 mb",  "HGT:500 mb",  "HGT:850 mb",  "HGT:1000 mb",
        "SPFH:50 mb", "SPFH:500 mb", "SPFH:850 mb", "SPFH:1000 mb",
        "TMP:50 mb",  "TMP:500 mb",  "TMP:850 mb",  "TMP:1000 mb",
        "UGRD:50 mb", "UGRD:500 mb", "UGRD:850 mb", "UGRD:1000 mb",
        "VGRD:50 mb", "VGRD:500 mb", "VGRD:850 mb", "VGRD:1000 mb",
    ]

    prs_regex = "|".join(prs_vars)

    FH = FastHerbie(
        dates_batch,
        model="hrrr",
        product="prs",
        fxx=[0],
        max_threads=8,
    )

    ds_prs = FH.xarray(prs_regex)

    ds_sfc = ensure_dataset(ds_sfc)
    ds_TMP = ensure_dataset(ds_TMP)
    ds_MSLMA = ensure_dataset(ds_MSLMA)
    ds_prs = ensure_dataset(ds_prs)

    t_batch_end = time.time()

    print(f"Batch read time: {t_batch_end - t_batch_start:.2f} seconds")

    process_batch(
        ds_sfc,
        ds_prs,
        ds_TMP,
        ds_MSLMA,
        dates_batch
    )

    del ds_sfc, ds_prs, ds_TMP, ds_MSLMA
    gc.collect()


t_global_end = time.time()
print(f"Total runtime: {(t_global_end - t_global_start) / 60:.2f} minutes")



Total hours to process: 120

Processing batch 1
Time range: From 2021-01-01 00:00:00 to 2021-01-01 23:00:00


C:\Users\chenk\Desktop\GSRCONUS\.venv\Lib\site-packages\herbie\core.py:1280: UserWarning: Will not remove GRIB file because it previously existed.
  warnings.warn("Will not remove GRIB file because it previously existed.")


Batch read time: 249.26 seconds
write: output/2021/01/01/00.parquet
write: output/2021/01/01/01.parquet
write: output/2021/01/01/02.parquet
write: output/2021/01/01/03.parquet
write: output/2021/01/01/04.parquet
write: output/2021/01/01/05.parquet
write: output/2021/01/01/07.parquet
write: output/2021/01/01/08.parquet
write: output/2021/01/01/09.parquet
write: output/2021/01/01/10.parquet
write: output/2021/01/01/11.parquet
write: output/2021/01/01/12.parquet
write: output/2021/01/01/13.parquet
write: output/2021/01/01/14.parquet
write: output/2021/01/01/15.parquet
write: output/2021/01/01/16.parquet
write: output/2021/01/01/17.parquet
write: output/2021/01/01/18.parquet
write: output/2021/01/01/19.parquet
write: output/2021/01/01/20.parquet
write: output/2021/01/01/21.parquet
write: output/2021/01/01/22.parquet
write: output/2021/01/01/23.parquet

Processing batch 2
Time range: From 2021-01-02 00:00:00 to 2021-01-02 23:00:00
Batch read time: 264.26 seconds
write: output/2021/01/02/00.